### Chapter 10: Tool Engineering
### Topic 3: Designing `validate_fd_reference` Properly

### End-to-End Flow of Designing a Good Tool Interface

This chapter explains how to design a tool that returns information clearly, consistently, and safely.

The program performs the following steps in order:

1. The LLM requests a tool.
2. The tool receives the input.
3. The tool performs the lookup.
4. The tool prepares a structured result.
5. The application returns the result to the LLM.
6. The LLM uses that structured result to continue reasoning.

The focus of this chapter is **not how the lookup happens**, but **how the tool should communicate its result**.

---

### Why Does Tool Design Matter?

Suppose the customer asks:

> **"Check my FD reference FD123456."**

The LLM requests:

```text
validate_fd_reference("FD123456")
```

The tool searches the database.

Now the question becomes:

> **What should the tool return?**

Should it return:

```text
Success
```

or

```text
True
```

or

```text
Reference Valid
```

These answers are not sufficient.

The LLM needs enough information to decide what to do next.

This is why designing the tool's output is extremely important.

---

### Why Isn't Success or Failure Enough?

Suppose the tool returns:

```text
False
```

What does that mean?

It could mean:

- The customer typed the wrong reference number.
- The reference does not exist.
- The database is temporarily unavailable.
- The lookup failed.
- The customer is unauthorized.

All these situations return the same value:

```text
False
```

The LLM cannot understand what actually happened.

A good tool should return **business information**, not just technical success or failure.

---

### Why Is `found = False` Important?

Suppose the customer enters:

```text
FD999999
```

The tool searches the database.

No record exists.

Instead of returning:

```text
False
```

the tool returns:

```text
found = False

reference_number = FD999999
```

Now the LLM clearly understands:

> The lookup worked successfully.

> The reference number simply does not exist.

Notice something important.

This is **not an error**.

The tool worked perfectly.

The customer just provided a reference number that was not found.

---

### What About `found = True`?

Suppose the customer enters:

```text
FD123456
```

The tool finds a record.

Now it returns:

```text
found = True

reference_number = FD123456
```

This tells the LLM:

- The lookup succeeded.
- A matching record exists.
- More information is available.

Now the LLM can continue reasoning.

---

### Why Are These Two Cases Different?

Consider these situations.

Case 1

```text
found = False
```

Meaning:

The customer entered a reference number that does not exist.

---

Case 2

```text
found = True

status = Closed
```

Meaning:

The reference exists.

The FD is already closed.

These are completely different business situations.

If both returned:

```text
False
```

the LLM would lose important information.

---

### Why Return Individual Fields?

Suppose the database contains:

```text
Customer

Rahul Sharma

Status

Active

Amount

₹5,00,000

Maturity Date

15-Jan-2028
```

Should the tool return:

```text
Rahul Sharma has an active FD of ₹5,00,000 maturing on 15-Jan-2028.
```

No.

Instead it returns:

```text
customer_name = Rahul Sharma

status = Active

fd_amount = ₹5,00,000

maturity_date = 15-Jan-2028
```

Each field is returned separately.

---

### Why Is Structured Output Better?

Suppose the LLM wants only the maturity date.

With structured output:

```text
maturity_date
```

is immediately available.

With a summary sentence:

```text
Rahul Sharma has an active FD...
```

the LLM (or another program) must first understand the sentence and extract the date.

Structured data is much easier to use.

---

### Why Not Return the Entire Database Row?

Suppose the database contains:

```text
Customer Name

PAN Number

Internal Customer ID

Branch Code

Created Timestamp

Last Updated

Audit Status

FD Amount
```

Does the LLM need all these fields?

No.

Most are internal implementation details.

Instead, the tool should return only the information required for its purpose.

Example:

```text
customer_name

status

fd_amount

maturity_date
```

This keeps the interface simple and secure.

---

### Why Echo the Input?

Suppose the tool returns:

```text
found = False
```

Which reference number was searched?

We don't know.

Instead the tool returns:

```text
reference_number = FD999999

found = False
```

Now the result is self-contained.

Anyone reading the result immediately knows which request it belongs to.

This becomes even more useful when multiple tools are executed during the same conversation.

---

### Why Return a Dictionary?

The tool returns something like:

```python
{
    "found": True,
    "reference_number": "FD123456",
    "customer_name": "Rahul Sharma",
    "status": "Active",
    "fd_amount": 500000,
    "maturity_date": "2028-01-15"
}
```

A dictionary is ideal because:

- Every field has a name.
- The LLM can read it easily.
- It can be converted to JSON.
- It can be sent directly inside a `tool_result`.

Simple structured data is easier to process than formatted text.

---

### Good Tool Design Principles

A production tool should:

- Return structured data.
- Return business information instead of simple success or failure.
- Clearly distinguish different business outcomes.
- Return only the required fields.
- Echo the input when useful.
- Keep the response simple and serializable.

---

### Production Engineer's Perspective

Think of a tool as an API.

The LLM is the client calling that API.

A well-designed API should:

- Be easy to understand.
- Return consistent responses.
- Clearly distinguish different outcomes.
- Avoid exposing unnecessary internal details.
- Return structured data that is easy for both the LLM and the application to process.

The better the tool interface, the easier it becomes for the LLM to reason correctly and for downstream systems to verify, log, and audit the results.

---

# End-to-End Flow of Tool Implementation

This chapter explains how `validate_fd_reference()` is implemented.

The previous chapter explained **what the tool should return**. This chapter explains **how the tool produces that result**.

The program performs the following steps in order:

1. The LLM requests the tool.
2. The application calls `validate_fd_reference()`.
3. The tool validates and cleans the input.
4. The tool searches the database.
5. The tool checks whether a record exists.
6. The tool creates a structured response.
7. The application sends the result back to the LLM.

---

### Where Does Tool Execution Start?

Suppose the LLM requests:

```text
validate_fd_reference()

reference_number = FD123456
```

The application receives this request.

It then executes the real Python function.

```python
validate_fd_reference("FD123456")
```

From this point onward, everything happens in normal Python code.

The LLM is no longer involved.

---

### Why Doesn't the Tool Read the Database Directly?

A common design is:

```python
validate_fd_reference()

↓

Read Database

↓

Return Result
```

This works, but it tightly couples the tool to the database.

Instead, our project separates responsibilities.

```text
validate_fd_reference()

↓

get_fd_record()

↓

Database
```

---

### What Is `get_fd_record()`?

`get_fd_record()` is responsible only for reading data.

Its job is simple:

- Receive a reference number.
- Search the database.
- Return the matching record.

It does **not** know:

- Who called it.
- Whether the caller is an LLM.
- Whether Tool Calling is being used.

It is simply a reusable database access function.

---

### Why Separate These Two Functions?

Suppose tomorrow another tool needs customer details.

Example:

```python
get_customer_details()
```

It can also use:

```python
get_fd_record()
```

without copying database logic.

This separation improves:

- Reusability
- Testing
- Maintenance
- Code readability

---

### What Does `validate_fd_reference()` Actually Do?

Its responsibility is different.

Instead of reading the database, it focuses on business logic.

Example:

- Clean the input.
- Call `get_fd_record()`.
- Decide whether the record exists.
- Build the final response.
- Return structured data.

It acts as a bridge between the LLM and the database.

---

### Why Clean the Input?

Suppose the LLM extracts:

```text
FD123456
```

But the original email contained:

```text
FD123456
```

with extra spaces.

Example:

```text
"  FD123456  "
```

If the database searches for this value exactly,

it may fail.

So the tool performs:

```python
reference_number.strip()
```

Now the lookup uses:

```text
FD123456
```

This removes accidental whitespace without changing the actual reference number.

---

### Why Don't We Aggressively Modify Inputs?

Suppose the customer enters:

```text
FD12ABC
```

Should the tool automatically change it?

No.

Over-correcting user input can hide genuine mistakes.

The tool should only perform safe cleanup, such as removing leading and trailing spaces.

Business validation should remain explicit.

---

### How Does the Database Search Work?

The cleaned reference number is passed to:

```python
get_fd_record(reference_number)
```

The database performs an **exact lookup**.

Example:

```text
FD123456
```

matches only:

```text
FD123456
```

It does **not** search for similar reference numbers.

This is important because customer identifiers must always use exact matching.

---

### What Happens If No Record Exists?

Suppose the database returns:

```text
No Match
```

The tool creates:

```python
{
    "found": False,
    "reference_number": "FD999999"
}
```

Notice something important.

The database lookup succeeded.

The reference number simply does not exist.

This is a normal business outcome—not a system failure.

---

### What Happens If a Record Exists?

Suppose the database returns:

```text
Customer

Rahul Sharma

Status

Active

Amount

₹5,00,000
```

The tool creates:

```python
{
    "found": True,
    "reference_number": "FD123456",
    "customer_name": "Rahul Sharma",
    "status": "Active",
    "fd_amount": 500000,
    "maturity_date": "2028-01-15"
}
```

The tool converts the raw database record into a clean interface for the LLM.

---

### Why Doesn't the Tool Return the Raw Database Row?

A database record may contain:

```text
Customer Name

PAN

Branch Code

Internal Customer ID

Created Timestamp

Updated Timestamp

FD Amount

Status
```

Many of these fields are unnecessary.

Some may even be confidential.

The tool should expose only the information required by the LLM.

This keeps the interface:

- Simple
- Stable
- Secure

---

### Why Return a Dictionary?

The tool returns a Python dictionary.

Example:

```python
{
    "found": True,
    "customer_name": "Rahul Sharma",
    "status": "Active"
}
```

The application can easily:

- Convert it to JSON.
- Send it as a `tool_result`.
- Log it.
- Validate it.

Structured data is much easier to process than plain text.

---

### How Does This Connect to Tool Calling?

The execution flow becomes:

```text
LLM

↓

Requests validate_fd_reference()

↓

Application

↓

Calls validate_fd_reference()

↓

validate_fd_reference()

↓

Calls get_fd_record()

↓

Database

↓

Returns Record

↓

validate_fd_reference()

↓

Creates Structured Result

↓

Application

↓

Sends tool_result to LLM
```

Notice that the LLM never interacts with the database directly.

Everything passes through the tool.

---

### Why Is This Design Useful?

This separation allows each component to focus on one responsibility.

**LLM**

- Decides which tool to use.

**Application**

- Executes the tool.

**validate_fd_reference()**

- Applies business logic.

**get_fd_record()**

- Reads the database.

**Database**

- Stores customer records.

Each component has a single responsibility.

This makes the system easier to understand, test, and maintain.

---

### Production Engineer's Perspective

A production tool should never mix multiple responsibilities.

Instead, separate the layers:

- The LLM decides **what information is needed**.
- The application decides **which tool to execute**.
- The tool applies **business logic**.
- The database access function retrieves **raw data**.
- The tool converts raw data into a structured response.
- The application returns that response to the LLM.

This layered architecture keeps the code modular, reusable, secure, and easy to maintain as the system grows.

---

# End-to-End Flow of Production Tool Engineering

This chapter explains how a production tool should behave after it has been implemented.

A tool that works correctly is not automatically production-ready. It must also be reliable, secure, easy to debug, and easy to maintain.

The program performs the following steps in order:

1. Receive the tool request.
2. Validate the input.
3. Execute the lookup.
4. Return a structured response.
5. Log the request and response.
6. Monitor tool performance.
7. Handle errors correctly.
8. Continue the agent workflow.

---

### What Is a Successful Tool?

Many developers think:

> "The tool returned the correct answer."

So the tool is finished.

In production, that is only the beginning.

A production tool should also be:

- Reliable
- Secure
- Easy to debug
- Easy to monitor
- Easy to extend

---

### How Should a Tool Handle "Not Found"?

Suppose the customer enters:

```text
FD999999
```

The reference number does not exist.

Should the tool throw an exception?

No.

This is not a software error.

It is a normal business outcome.

Instead, return:

```python
{
    "found": False,
    "reference_number": "FD999999"
}
```

Now the LLM can politely tell the customer that the reference number was not found.

---

### When Should a Tool Throw an Exception?

Now suppose the database server is down.

The tool cannot perform the lookup.

This is different.

Example:

```text
Database Connection Failed
```

This is a genuine system failure.

The application should:

- Log the error.
- Retry if appropriate.
- Inform the user gracefully.
- Alert engineers if necessary.

Notice the difference.

**Business outcome**

```text
Reference Not Found
```

is expected.

**System failure**

```text
Database Unavailable
```

is unexpected.

Production systems treat these differently.

---

### Why Is Logging Important?

Every tool execution should be logged.

Example:

```text
Timestamp

10:35:42

------------------------

Tool

validate_fd_reference

------------------------

Input

FD123456

------------------------

Result

found = True

------------------------

Execution Time

18 ms
```

Logs help answer questions like:

- What tool was executed?
- Which customer record was requested?
- How long did the lookup take?
- What result was returned?

Without logs, debugging becomes extremely difficult.

---

### What Should We Monitor?

Production teams monitor every tool.

Typical metrics include:

```text
Total Calls

Success Rate

Failure Rate

Average Latency

Maximum Latency

Database Errors

Timeouts
```

Example:

```text
validate_fd_reference

Calls

12,000

Average Latency

14 ms

Failure Rate

0.1%
```

These metrics help detect production issues early.

---

### Why Is Structured Output Better for Monitoring?

Suppose the tool returns:

```text
Customer Rahul Sharma has an active FD of ₹5,00,000.
```

Monitoring systems now have to understand English.

Instead, suppose the tool returns:

```python
{
    "found": True,
    "status": "Active",
    "fd_amount": 500000
}
```

Now dashboards can easily calculate:

- Active FDs
- Closed FDs
- Average FD amount

Structured output benefits both the LLM and operational monitoring.

---

### Why Should We Avoid Returning Raw Database Records?

Suppose the database contains:

```text
PAN Number

Customer Address

Internal Customer ID

Audit Columns

Branch Code
```

Does the LLM need these fields?

No.

Returning unnecessary fields creates risks:

- Information leakage
- Larger payloads
- Higher token usage
- Increased maintenance

Always expose only the fields required for the task.

This follows the **Principle of Least Privilege**.

---

### Why Is Input Validation Important?

Suppose the LLM sends:

```text
reference_number = ""
```

or

```text
reference_number = FD12XYZ
```

The tool should validate:

- Is the input present?
- Is the format correct?
- Is the length valid?

Invalid inputs should never reach the database.

This reduces unnecessary database queries and improves security.

---

### Why Does Latency Matter?

Suppose:

```text
validate_fd_reference()

15 ms
```

and

```text
search_knowledge_base()

600 ms
```

Both are tool calls.

But they have very different performance characteristics.

A simple indexed lookup is usually very fast.

Semantic search is significantly more expensive.

Knowing the latency of each tool helps optimize the overall agent.

---

### Why Does Cost Matter?

Some tools are almost free.

Example:

```text
Database Lookup
```

Other tools are expensive.

Example:

```text
Large Knowledge Base Search

External APIs

LLM Calls
```

Production engineers try to:

- Avoid unnecessary tool calls.
- Cache repeated lookups.
- Reuse results whenever possible.

Reducing unnecessary tool executions lowers both latency and cost.

---

### How Should We Debug a Tool?

When something goes wrong, determine where the failure occurred.

Possible locations:

**Input**

```text
Invalid Reference Number
```

---

**Tool**

```text
Validation Logic Failed
```

---

**Database**

```text
Connection Error
```

---

**Application**

```text
Dispatch Layer Failed
```

---

**LLM**

```text
Requested Wrong Tool
```

Finding the correct failure point is much faster than assuming the LLM is responsible.

---

### What Are the Design Trade-offs?

#### Return Structured Data or Summary Text?

Preferred:

```python
{
    "status": "Active",
    "amount": 500000
}
```

Reason:

Easy for both applications and LLMs to process.

---

#### Return Selected Fields or Entire Database Row?

Preferred:

Selected fields.

Reason:

- Smaller responses.
- Better security.
- Lower token usage.
- Simpler interface.

---

#### Return "Not Found" or Raise Exception?

Preferred:

```text
found = False
```

Reason:

A missing customer reference is a normal business outcome.

Exceptions should represent genuine system failures.

---

#### Where Should Input Validation Happen?

Possible locations:

- Dispatch Layer
- Tool Function

Production systems often validate in both places.

The dispatcher performs basic validation.

The tool performs business validation.

This provides defense in depth.

---

### Common Production Mistakes

#### Mistake 1

Returning only:

```text
Success
```

or

```text
Failure
```

The LLM loses important business information.

---

#### Mistake 2

Returning one long English sentence.

Structured output is much easier to process.

---

#### Mistake 3

Returning raw database records.

This may expose confidential internal information.

---

#### Mistake 4

Treating "Not Found" as a software error.

It is usually a normal business outcome.

---

#### Mistake 5

Skipping logging.

Without logs, production debugging becomes very difficult.

---

#### Mistake 6

Trusting model inputs without validation.

Always validate before executing the tool.

---

### Production Engineer's Perspective

A production tool is much more than a Python function.

It should:

- Validate inputs.
- Return structured outputs.
- Clearly distinguish business outcomes from system failures.
- Return only the required information.
- Log every execution.
- Expose useful monitoring metrics.
- Protect confidential data.
- Keep latency low.
- Minimize unnecessary database calls.
- Be modular and reusable.

A useful mental model is:

```text
Tool

↓

Validate Input

↓

Execute Business Logic

↓

Retrieve Data

↓

Create Structured Result

↓

Log

↓

Monitor

↓

Return to LLM
```

A well-designed production tool is predictable, secure, easy to maintain, and easy for both humans and LLMs to reason about.

---

### Lead-Level Interview Questions

**Basic**

- Q: Why does `validate_fd_reference` return a structured `found: False` result instead of raising an exception when a reference number doesn't exist?  
  A: A customer providing an incorrect or typo'd reference number is a routine, expected business outcome, not a system error — treating it as ordinary structured data lets the calling agent reason about and handle it as a normal case (for example, asking the customer to double-check the number), rather than conflating it with a genuine system failure that an exception would typically represent.

- Q: Why does the tool return several individually-named fields rather than one combined summary string?  
  A: Structured, individually-addressable fields let downstream code (and the model itself) reference exactly the specific piece of information it needs, and make precise verification — like checking whether a specific cited amount matches the record — straightforward. A single flattened string would require re-parsing to extract any specific fact, which is fragile and error-prone.

**Intermediate**

- Q: What's the significance of the docstring's explicit statement that "found=False... is meaningfully different from 'exists but something's off'"?  
  A: It documents a deliberate design decision: this tool is built to distinguish between genuinely different outcomes rather than collapsing them into a single notion of "invalid." This distinction matters because the correct downstream behavior differs — a nonexistent reference number likely warrants asking the customer to recheck it, while a reference number that exists but has an unusual status might warrant a different kind of handling or escalation entirely.

- Q: Why does the tool echo the original `reference_number` back in its result, even when nothing was found?  
  A: This makes the tool's result self-contained and traceable — the calling code (and, in a multi-tool scenario, potentially the model reasoning over several results at once) doesn't need to separately track which specific input a given result corresponds to, since the result carries that information along with it.

**Advanced**

- Q: A teammate proposes simplifying `validate_fd_reference` by having it return the entire raw database row directly, rather than a curated set of named fields. What would you push back on?  
  A: Returning the raw row exposes internal implementation details — actual column names, internal-only fields never meant for the model or downstream consumption, and the underlying data structure's specific shape — that the tool's design should otherwise encapsulate. A curated, purpose-built field set is a deliberate interface choice: it exposes exactly what the model needs to reason correctly, and nothing more, making the tool more maintainable (the underlying database schema can change without necessarily changing the tool's contract) and more secure (nothing unintended leaks through).

- Q: How does this tool's design specifically support the faithfulness and citation verification work from Chapter 8?  
  A: Faithfulness checking (Chapter 8 Topic 3/4) needs to verify specific claims against specific pieces of retrieved or looked-up information — this requires that information to be structured and individually addressable. Because `validate_fd_reference` returns distinct fields like `fd_amount_inr` and `status` rather than a single prose summary, a downstream check can directly compare "the answer claims Rs 391,000" against "the record's `fd_amount_inr` field says 391000" without needing to parse meaning back out of a flattened string — this tool's output shape is precisely what makes that kind of precise, field-level verification tractable at all.

**Scenario-based**

- Q: A production incident report shows the agent gave a customer an answer treating their genuinely nonexistent FD reference number as if it were a valid, closed account. Walk through your investigation, focusing on this tool's design.  
  A: First check whether `validate_fd_reference` actually returned `found: False` correctly for this specific reference number, or whether it incorrectly matched to a different record — if the tool's own found/not-found logic is working correctly, the bug more likely lies in how the calling agent or the generation layer interpreted or handled a `found: False` result, rather than in the tool itself. This is exactly why the found/not-found distinction needs to be unambiguous and thoroughly tested at the tool level (this chapter's upcoming testing topic) — a well-designed tool result should make this kind of downstream misinterpretation harder to produce in the first place, not easier.

**Follow-up questions to expect:**

- "How would you extend this tool's design if the underlying data source changed from a CSV to a live production database?"
- "What additional fields, if any, would you consider adding to this tool's result, and how would you decide?"


### Module 1: The Real Tool, Rebuilt Faithfully

Rebuild `validate_fd_reference` and its underlying `get_fd_record` exactly matching the project's actual design -- separate layers, structured output, input sanitization.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: validate_fd_reference, rebuilt faithfully to the real
# project design -- separate layers, structured output, sanitization
# ------------------------------------------------------------------

FD_RECORDS_TABLE = {
    "BJ2019FD7717": {"Customer_Name": "Shobha Chopra", "Status": "Closed (Premature)",
                      "FD_Amount_INR": 391000, "Maturity_Date": "2024-03-15"},
    "BJ2022FD6918": {"Customer_Name": "Anita Mishra", "Status": "Active",
                      "FD_Amount_INR": 160000, "Maturity_Date": "2026-11-02"},
}

def get_fd_record(reference_number: str, path: str = None) -> dict:
    """Lower-level, reusable data-access function -- knows NOTHING
    about tools, schemas, or the model. Just a plain lookup, exactly
    mirroring the real project's separation of concerns."""
    return FD_RECORDS_TABLE.get(reference_number)  # None if not found


def validate_fd_reference(reference_number: str) -> dict:
    """The REAL tool-facing function, rebuilt faithfully:
    - sanitizes expected input noise (.strip())
    - distinguishes found=False from found=True explicitly
    - echoes the input back in EVERY result
    - returns structured, individually-named fields, never a
      flattened summary string"""
    clean_ref = reference_number.strip()
    record = get_fd_record(clean_ref)

    if record is None:
        return {"reference_number": reference_number, "found": False}

    return {
        "reference_number": reference_number,
        "found": True,
        "customer_name_on_record": record["Customer_Name"],
        "status": record["Status"],
        "fd_amount_inr": record["FD_Amount_INR"],
        "maturity_date": record["Maturity_Date"],
    }


print("=" * 70)
print("MODULE 1: THE REAL TOOL, REBUILT FAITHFULLY")
print("=" * 70)

test_cases = [
    ("BJ2019FD7717", "exact match, no noise"),
    ("  BJ2022FD6918  ", "valid reference WITH whitespace noise"),
    ("BJ9999FD0000", "genuinely nonexistent reference number"),
]

for ref, label in test_cases:
    result = validate_fd_reference(ref)
    print(f"\n[{label}]")
    print(f"  Input: '{ref}'")
    print(f"  Result: {result}")

print("\nNotice: whitespace noise did NOT prevent a valid match (sanitization")
print("working correctly), and the genuinely nonexistent reference correctly")
print("returned found=False WITH the original input echoed back -- exactly")
print("the structured, traceable design the theory describes.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: THE REAL TOOL, REBUILT FAITHFULLY

[exact match, no noise]
  Input: 'BJ2019FD7717'
  Result: {'reference_number': 'BJ2019FD7717', 'found': True, 'customer_name_on_record': 'Shobha Chopra', 'status': 'Closed (Premature)', 'fd_amount_inr': 391000, 'maturity_date': '2024-03-15'}

[valid reference WITH whitespace noise]
  Input: '  BJ2022FD6918  '
  Result: {'reference_number': '  BJ2022FD6918  ', 'found': True, 'customer_name_on_record': 'Anita Mishra', 'status': 'Active', 'fd_amount_inr': 160000, 'maturity_date': '2026-11-02'}

[genuinely nonexistent reference number]
  Input: 'BJ9999FD0000'
  Result: {'reference_number': 'BJ9999FD0000', 'found': False}

Notice: whitespace noise did NOT prevent a valid match (sanitization
working correctly), and the genuinely nonexistent reference correctly
returned found=False WITH the original input echoed back -- exactly
the structured, traceable design the theory describes.

Module 1 complete. Run Module 2 next.


### Module 2: Why the found/not-found Distinction Matters — A Concrete Failure Comparison

Build a DELIBERATELY WORSE version that collapses found/not-found into a single generic result, and show concretely what information gets lost for downstream reasoning.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Collapsed vs distinct found/not-found -- concrete comparison
# ------------------------------------------------------------------

def validate_fd_reference_BAD_DESIGN(reference_number: str) -> str:
    """A DELIBERATELY WORSE design -- collapses everything into one
    flattened string, with no way to programmatically distinguish
    'not found' from 'found but unusual' from 'found and normal'."""
    clean_ref = reference_number.strip()
    record = get_fd_record(clean_ref)
    if record is None:
        return "Could not find this FD reference."
    return f"Found: {record['Customer_Name']}, status {record['Status']}, amount {record['FD_Amount_INR']}"


print("=" * 70)
print("COLLAPSED (BAD) DESIGN vs STRUCTURED (REAL) DESIGN")
print("=" * 70)

test_ref_found = "BJ2019FD7717"
test_ref_not_found = "BJ9999FD0000"

bad_found = validate_fd_reference_BAD_DESIGN(test_ref_found)
bad_not_found = validate_fd_reference_BAD_DESIGN(test_ref_not_found)

good_found = validate_fd_reference(test_ref_found)
good_not_found = validate_fd_reference(test_ref_not_found)

print(f"\nBAD design output for FOUND case:     '{bad_found}'")
print(f"BAD design output for NOT-FOUND case: '{bad_not_found}'")

print(f"\nGOOD design output for FOUND case:     {good_found}")
print(f"GOOD design output for NOT-FOUND case: {good_not_found}")

# REAL, concrete test: can calling code programmatically check
# "was this found?" WITHOUT string-parsing, for each design?
def can_check_found_programmatically_bad(output: str) -> bool:
    """With the bad design, this requires FRAGILE STRING MATCHING --
    demonstrating exactly the problem the theory describes."""
    return "Could not find" not in output  # fragile: depends on exact wording

def can_check_found_programmatically_good(output: dict) -> bool:
    """With the good design, this is a direct, reliable field access."""
    return output["found"]

print(f"\nChecking 'was it found?' for the FOUND case:")
print(f"  BAD design requires string-matching:  {can_check_found_programmatically_bad(bad_found)}")
print(f"  GOOD design is a direct field access:  {can_check_found_programmatically_good(good_found)}")

print(f"\nWhat happens if the BAD design's wording ever changes slightly")
print(f"(e.g. 'Could not locate' instead of 'Could not find')?")
modified_bad_output = "Could not locate this FD reference."
print(f"  New wording: '{modified_bad_output}'")
print(f"  string-match check now returns: {can_check_found_programmatically_bad(modified_bad_output)}")
print(f"  (WRONG -- should be False/not-found, but the fragile string check")
print(f"   breaks silently the moment wording changes even slightly)")

print("\nThis is the CONCRETE, measurable cost of collapsing structured")
print("data into a flattened string: downstream code becomes fragile,")
print("silently breakable, and requires re-parsing meaning that was")
print("already available as real, structured data before it was flattened.")
print("\nModule 2 complete. Run Module 3 next.")


COLLAPSED (BAD) DESIGN vs STRUCTURED (REAL) DESIGN

BAD design output for FOUND case:     'Found: Shobha Chopra, status Closed (Premature), amount 391000'
BAD design output for NOT-FOUND case: 'Could not find this FD reference.'

GOOD design output for FOUND case:     {'reference_number': 'BJ2019FD7717', 'found': True, 'customer_name_on_record': 'Shobha Chopra', 'status': 'Closed (Premature)', 'fd_amount_inr': 391000, 'maturity_date': '2024-03-15'}
GOOD design output for NOT-FOUND case: {'reference_number': 'BJ9999FD0000', 'found': False}

Checking 'was it found?' for the FOUND case:
  BAD design requires string-matching:  True
  GOOD design is a direct field access:  True

What happens if the BAD design's wording ever changes slightly
(e.g. 'Could not locate' instead of 'Could not find')?
  New wording: 'Could not locate this FD reference.'
  string-match check now returns: True
  (WRONG -- should be False/not-found, but the fragile string check
   breaks silently the moment wording c

### Module 3: Traceability — Echoing the Input Back, Tested

Demonstrates concretely why echoing the input matters, using a multi-call scenario where results need to be matched back to their originating requests.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Traceability -- echoing the input back, tested concretely
# ------------------------------------------------------------------

# simulate a scenario with SEVERAL tool results arriving together
# (as would happen in a multi-tool-call turn, Topic 6's territory)
multiple_requests = ["BJ2019FD7717", "BJ9999FD0000", "BJ2022FD6918"]
results = [validate_fd_reference(ref) for ref in multiple_requests]

print("=" * 70)
print("TRACEABILITY: matching results back to their requests")
print("=" * 70)
print("Multiple results arrived together (simulating a multi-tool turn):\n")
for r in results:
    print(f"  {r}")

# REAL test: can we correctly match each result back to its request,
# using ONLY the result itself (no external tracking needed)?
print("\nMatching each result back to its original request, using ONLY")
print("the echoed 'reference_number' field inside each result itself:")
for original_ref in multiple_requests:
    matching_result = next(r for r in results if r["reference_number"] == original_ref)
    found_status = matching_result["found"]
    print(f"  Request '{original_ref}' -> matched result: found={found_status}")

print("\nThis worked because EVERY result carries its own originating input")
print("along with it -- no separate bookkeeping needed by the calling code")
print("to remember 'which result answers which request'. This becomes")
print("essential once Topic 6's multi-tool agents call several different")
print("tools, potentially with several instances of the SAME tool, in a")
print("single turn.")

print("\nModule 3 complete. All Chapter 10 Topic 3 modules done.")
print()
print("=" * 70)
print("CHAPTER 10 TOPIC 3 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  found=False vs found=True is a DELIBERATE, meaningful distinction --
  never collapse "doesn't exist" and "exists but unusual" into one
  generic failure state.

  Structured, individually-named fields (not a flattened summary
  string) are what make downstream faithfulness/citation checking
  (Chapter 8) tractable -- demonstrated concretely: string-matching a
  flattened output is fragile and breaks silently on wording changes.

  Sanitize EXPECTED noise (whitespace) inside the tool; don't over-reach
  into masking genuinely malformed input that should be validated and
  refused instead (Topic 1's validation-before-execution principle).

  ALWAYS echo the original input back in the result -- this is what
  makes results self-contained and traceable, critical once multiple
  tool calls are in flight in a single turn (Topic 6).

  Keep tool-facing shaping logic (validate_fd_reference) SEPARATE from
  general, reusable data access (get_fd_record) -- a clean layering
  principle that generalizes to every other tool in this project.
""")


TRACEABILITY: matching results back to their requests
Multiple results arrived together (simulating a multi-tool turn):

  {'reference_number': 'BJ2019FD7717', 'found': True, 'customer_name_on_record': 'Shobha Chopra', 'status': 'Closed (Premature)', 'fd_amount_inr': 391000, 'maturity_date': '2024-03-15'}
  {'reference_number': 'BJ9999FD0000', 'found': False}
  {'reference_number': 'BJ2022FD6918', 'found': True, 'customer_name_on_record': 'Anita Mishra', 'status': 'Active', 'fd_amount_inr': 160000, 'maturity_date': '2026-11-02'}

Matching each result back to its original request, using ONLY
the echoed 'reference_number' field inside each result itself:
  Request 'BJ2019FD7717' -> matched result: found=True
  Request 'BJ9999FD0000' -> matched result: found=False
  Request 'BJ2022FD6918' -> matched result: found=True

This worked because EVERY result carries its own originating input
along with it -- no separate bookkeeping needed by the calling code
to remember 'which result answers whi